In [1]:
# ==========================================
# Imports
# ==========================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from anova_module import ModelAnalysis

In [2]:
# ==========================================
# Processing
# ==========================================

# ==========================================
# 1. DOWNLOAD & PREPARATION
# ==========================================
print("Downloading Dota 2 dataset...")
dota2 = fetch_ucirepo(id=367)

# Retrieve as Pandas DataFrame
X = dota2.data.features
y = dota2.data.targets

print(f"Dimensions before cleaning: {X.shape}")

cols_to_drop = ['gamemode', 'gametype']

# First check if columns exist to avoid errors
existing_cols_to_drop = [col for col in cols_to_drop if col in X.columns]
X = X.drop(columns=existing_cols_to_drop)

print(f"Columns dropped: {existing_cols_to_drop}")
print(f"Dimensions after cleaning: {X.shape}")
# ---------------------------------------------------

# Target encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y.values.ravel())

# Normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Conversion to Tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=1024)

# ==========================================
# 2. MODEL ARCHITECTURE (Deep Funnel)
# ==========================================
class DotaMLP(nn.Module):
    def __init__(self, input_dim):
        super(DotaMLP, self).__init__()
        
        # Architecture: Input -> 1024 -> 512 -> 256 -> 128 -> 64 -> 32 -> 2
        
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.layer2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.layer3 = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.layer4 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.layer5 = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.layer6 = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )
        
        self.output = nn.Linear(32, 2)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        return self.output(x)

# Dynamic initialization (input_dim adapts automatically after column dropping)
input_dimension = X_train.shape[1] 
print(f"Network input dimension: {input_dimension}") 
model = DotaMLP(input_dimension).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 3. TRAINING
# ==========================================
def calculate_accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

print("\nStarting training...")
epochs = 10  

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    train_acc = calculate_accuracy(train_loader)
    test_acc = calculate_accuracy(test_loader)
    print(f"Epoch {epoch+1:02d} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2%} | Test Acc: {test_acc:.2%}")

# ==========================================
# 4. FINAL RESULT
# ==========================================
final_acc = calculate_accuracy(test_loader)
print(f"\n>>> Final Accuracy: {final_acc:.2%}")

Dimensions before cleaning: (102944, 115)
Columns dropped: ['gamemode', 'gametype']
Dimensions after cleaning: (102944, 113)
Training on: cpu
Network input dimension: 113

Starting training...
Epoch 01 | Loss: 0.6791 | Train Acc: 60.45% | Test Acc: 60.02%
Epoch 02 | Loss: 0.6653 | Train Acc: 61.12% | Test Acc: 60.09%
Epoch 03 | Loss: 0.6618 | Train Acc: 61.89% | Test Acc: 60.12%
Epoch 04 | Loss: 0.6588 | Train Acc: 62.49% | Test Acc: 60.12%
Epoch 05 | Loss: 0.6559 | Train Acc: 63.38% | Test Acc: 59.58%
Epoch 06 | Loss: 0.6517 | Train Acc: 63.80% | Test Acc: 59.04%
Epoch 07 | Loss: 0.6475 | Train Acc: 65.03% | Test Acc: 59.19%
Epoch 08 | Loss: 0.6439 | Train Acc: 65.38% | Test Acc: 59.56%
Epoch 09 | Loss: 0.6399 | Train Acc: 66.26% | Test Acc: 58.91%
Epoch 10 | Loss: 0.6335 | Train Acc: 67.26% | Test Acc: 58.69%

>>> Final Accuracy: 58.69%


In [3]:
# ==========================================
# Function
# ==========================================

X_numpy = X.to_numpy()
r , d = X_numpy.shape

# Function of interest (proba class 0)
def f_model(X_numpy):
    if hasattr(scaler, 'feature_names_in_'):
        X_input = pd.DataFrame(X_numpy, columns=scaler.feature_names_in_)
    else:
        X_input = X_numpy
    
    X_numpy_scaled = scaler.transform(X_input)
    
    X_tensor = torch.tensor(X_numpy_scaled, dtype=torch.float32).to(device)
    
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)      
        probs = torch.softmax(logits, dim=1)
        
        predictions_class_1 = probs[:, 0]

    return predictions_class_1.cpu().numpy()

In [4]:
%%time
# =============================================
# Functional ANOVA Decomposition (MAIN EFFECTS)
# =============================================

A = ModelAnalysis(X_numpy , f_model , 0.109 , 1 , 1e-4) # percentage = 0.109 to have exactly all main effects
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 112/112 [00:04<00:00, 27.23it/s]


Computations complete. Results ready.
0.35835056703582835 0.015949335255252536 0.0621960291481925
CPU times: user 1min 7s, sys: 10.2 s, total: 1min 17s
Wall time: 23.7 s


In [5]:
%%time
# ==========================================
# Functional ANOVA Decomposition
# ==========================================

A = ModelAnalysis(X_numpy , f_model , 4 , 1e-2 , 1e-2)
S , Matrix = A.functional_anova() # sets and f_A(X_A)
print(A.get_R2() , A.get_L2_Error() , A.get_L2_Error_rel())

Constructing Basis Matrix: 100%|██████████| 4117/4117 [31:05<00:00,  2.21it/s]


Computations complete. Results ready.
0.4137793410121702 0.01457155472040142 0.05682323605468777
CPU times: user 58min 30s, sys: 20min 32s, total: 1h 19min 3s
Wall time: 38min 48s
